# Energy Dataset: Selecting a Polynomial Regression Degree
This notebook uses grid search to choose a polynomial regression degree for predicting electrical power output.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from scipy.stats import randint

## Data Inspection
The energy dataset is loaded, the target variable is separated, and the feature columns are inspected.


In [2]:
df = pd.read_csv("../data/energy.csv")
X = df.drop("PE", axis=1)
y = df["PE"]
X.info()
df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9568 entries, 0 to 9567
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   AT      9564 non-null   float64
 1   V       9568 non-null   float64
 2   AP      9568 non-null   float64
 3   RH      9568 non-null   float64
dtypes: float64(4)
memory usage: 299.1 KB


AT    4
V     0
AP    0
RH    0
PE    0
dtype: int64

## Train-Test Split
The data is split into training and test sets before preprocessing and model selection.


In [13]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Preprocessing
Numerical features are imputed and standardized as part of the modeling pipeline.


In [14]:
num_pipeline = make_pipeline(
    SimpleImputer(strategy="mean"),
    StandardScaler()
)

attributes = list(X_train.columns)
preprocessing = ColumnTransformer([
    ("num", num_pipeline, attributes)
])

## Model Selection
Grid search compares polynomial feature degrees for a linear regression pipeline.


In [22]:
reg = make_pipeline(
    preprocessing,
    PolynomialFeatures(),
    LinearRegression()
)

param_grid = {
    "polynomialfeatures__degree": [1,3,5,7]
}

gridsearch = GridSearchCV(reg, param_grid, scoring='r2', cv=5)
res = gridsearch.fit(X_train, y_train)

print("Best model configuration is:")
print(res.best_params_)
print("with R2=%.2f" % res.best_score_)

Best model configuration is:
{'polynomialfeatures__degree': 5}
with R2=0.94


## Evaluation
The selected polynomial model is evaluated on the test set with R2, MSE, RMSE, and MAE.


In [23]:
best_model = gridsearch.best_estimator_
y_pred = best_model.predict(X_test)

print("Test R2:", r2_score(y_test, y_pred))
print("Test MSE:", mean_squared_error(y_test, y_pred))
print("Test RMSE:", mean_squared_error(y_test, y_pred) ** 0.5)
print("Test MAE:", mean_absolute_error(y_test, y_pred))

Test R2: 0.9459106864029431
Test MSE: 15.477339405085004
Test RMSE: 3.9341249859511334
Test MAE: 3.0305388489387024
